In [2]:
# 1. Importamos librerías
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

print("Librerías cargadas correctamente.")

# 2. Cargamos los datos
df = pd.read_csv('../data/raw/churn_data.csv')

# Separamos las variables independientes (X) de la dependiente (y)
X = df.drop('Exited', axis=1)
y = df['Exited']

# 3. Preparamos los datos
# Identificamos que columnas son categóricas y cuáles son números
cat_cols = ['Geography', 'Gender']
num_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

# Creamos un preprocesador que escala los números y convierte el texto a binarias (OneHot)
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols )
    ]
)

# Dividimos en Train y Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Configuramos MLflow (Tracking)
mlflow.set_experiment("Bank_Churn_Prediction_Final_Project")

# 5. Definimos los 3 modelos a comparar
modelos_a_probar = {
    "Logistic_Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "XGBoost":XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=3)
}

mejor_f1 = 0
nombre_campeon = ""

print(" Inicamos los experimentos")

for nombre_modelo, clasificador in modelos_a_probar.items():
    # Iniciamos un 'run' separado en MLflow para cada modelo
    with mlflow.start_run(run_name=nombre_modelo):

        # Creamos el pipeline para este modelo en particular
        pipeline_actual = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', clasificador)
        ])

        # Entrenamos
        pipeline_actual.fit(X_train, y_train)

        # Predecimos
        y_pred = pipeline_actual.predict(X_test)

        # Calculamos F1_Score
        f1 =  f1_score(y_test, y_pred)

        print(f"{nombre_modelo} entrenado. F1-Score:{f1:.4f}")

        # Registramos en MLflow
        mlflow.log_param("model_type", nombre_modelo)
        mlflow.log_metric("f1_score", f1)
        mlflow.sklearn.log_model(pipeline_actual, "model")

        # Lógica para elegir al campeón
        if f1 > mejor_f1:
            mejor_f1 = f1
            nombre_campeon = nombre_modelo
            mejor_pipeline = pipeline_actual

print(f"\n El modelo campeón definitivo es: {nombre_campeon} con un F1-Score de {mejor_f1:.4f}")

print("\n REPORTE DETALLADO DEL CAMPEÓN")
# Usamos el pipeline completo que ganó para realizar las predicciones
y_pred_campeon = mejor_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_campeon))


Librerías cargadas correctamente.
 Inicamos los experimentos
Logistic_Regression entrenado. F1-Score:0.4991


2026/02/25 17:04:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 17:04:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/25 17:04:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 17:04:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Random_Forest entrenado. F1-Score:0.5629


2026/02/25 17:04:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 17:04:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost entrenado. F1-Score:0.5823

 El modelo campeón definitivo es: XGBoost con un F1-Score de 0.5823

 REPORTE DETALLADO DEL CAMPEÓN
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1593
           1       0.57      0.60      0.58       407

    accuracy                           0.82      2000
   macro avg       0.73      0.74      0.74      2000
weighted avg       0.83      0.82      0.83      2000

